In [ ]:
import numpy as np, cv2, torch, tensorflow as tf, sklearn, matplotlib.pyplot as mpl, mediapipe as mp, importlib.resources as res
from tensorflow.keras import models
from collections import deque
from symspellpy import SymSpell
from itertools import product

#TODO
#Ability to decode movements (3D CNN)
#Honestly train a better Model for static images. (3D CNN for both static and dynamic?)
#Camera with more than 20fps
#Refine TemporalLetterTracking to detect spaces
#Beginners and Experts will have different hand movement speeds(Remove HandMotionDetector caveats if individual can sign fast enough?)

In [13]:
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary(
    str(res.files("symspellpy") / "frequency_dictionary_en_82_765.txt"), 0, 1)
sym_spell.load_bigram_dictionary(
    str(res.files("symspellpy") / "frequency_bigramdictionary_en_243_342.txt"), 0, 2)

True

In [3]:
model = tf.keras.models.load_model('./MEDIAPIPE2_Model6.keras')

In [4]:
def hand_bbox(hand_landmarks, image_w, image_h, pad=0.25):
    xs = np.array([lm.x for lm in hand_landmarks.landmark]) * image_w
    ys = np.array([lm.y for lm in hand_landmarks.landmark]) * image_h

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    bw, bh = max(x_max - x_min, y_max - y_min), max(x_max - x_min, y_max - y_min)
    x_min -= bw * pad; 
    x_max += bw * pad
    y_min -= bh * pad; 
    y_max += bh * pad

    x_min = int(max(0, x_min)); 
    y_min = int(max(0, y_min))
    x_max = int(min(image_w, x_max)); 
    y_max = int(min(image_h, y_max))
    return x_min, y_min, x_max, y_max

In [5]:
class TemporalLetterTracking:
    """
    Tracks the accumulated probability of letters over a short window of time.
    """
    def __init__(self, labels, window=10, min_conf=0, top_k=1):
        self.labels = list(labels)
        self.buf = deque(maxlen=window)
        self.min_conf = min_conf
        self.top_k = top_k

    def update(self, preds):
        preds = np.asarray(preds).ravel()
        frame = {letter: float(v) for letter, v in zip(self.labels, preds)}

        if max(frame.values()) < self.min_conf:
            return None

        self.buf.append(frame)

        if len(self.buf) < self.buf.maxlen:
            return None

        totals = {letter: 0.0 for letter in self.labels}
        for f in self.buf:
            for letter, v in f.items():
                totals[letter] += v

        ranked = sorted(totals, key=totals.get, reverse=True)[:self.top_k]
        return [(letter, totals[letter]) for letter in ranked]

    def reset(self):
        self.buf.clear()
        

In [6]:
class HandMotionDetector:
    """
    Detects when the hand changes shape to form a new letter.
    Movement away from and towards the camera shouldn't trigger movement.
    """
    def __init__(self, move_thresh=0.05, still_thresh=0.06,
                 still_frames=5, smoothing=0.5):
        self.move_thresh = move_thresh
        self.still_thresh = still_thresh
        self.still_frames = still_frames
        self.smoothing = smoothing
        self.prev = None
        self.motion = 0.0
        self.state = "still" 
        self.still_count = 0

    @staticmethod
    def _normalize(hand_landmarks):
        pts = np.array([[lm.x, lm.y] for lm in hand_landmarks.landmark])
        pts = pts - pts[0]
        scale = np.linalg.norm(pts[9])
        return pts / scale if scale > 1e-6 else pts 

    def update(self, hand_landmarks):
        FINGER_JOINTS = [2,3,4, 6,7,8, 10,11,12, 14,15,16, 18,19,20]
        curr = self._normalize(hand_landmarks)
        if self.prev is None:
            self.prev = curr
            return False
    
        diffs = np.linalg.norm(curr[FINGER_JOINTS] - self.prev[FINGER_JOINTS], axis=1)
        raw = np.mean(diffs) 
        self.motion = self.smoothing * self.motion + (1 - self.smoothing) * raw
        self.prev = curr

        reset = False
        if self.state == "still":
            if self.motion > self.move_thresh:
                self.state = "moving"
                self.still_count = 0
                reset = True
        else:
            if self.motion < self.still_thresh:
                self.still_count += 1
                if self.still_count >= self.still_frames:
                    self.state = "still"
            else:
                self.still_count = 0
        return reset

    def accumulating(self):
        return self.state == "still"

In [7]:
def LetterMade(predictions, is_moving, threshold=7.6):
    """
    Returns an array when the hand is still and confident enough, else None.
    """
    if is_moving or predictions is None:
        return None
    top_total = predictions[0][1]
    if top_total > threshold:
        return predictions
    return None

In [14]:
def ReconstructSentence(clumped_text):
    return sym_spell.word_segmentation(clumped_text).corrected_string

In [9]:
confusion_dict = {'o':['c'], 'q':['p'], 'p':['q'], 'e':['s'], 'r':['u']}

def WordVariants(word):
    """
    Checks letters that the model still confuses. Unnecessary for more accurate model.
    """
    choices = [[c] + confusion_dict.get(c, []) for c in word]
    variants = {''.join(combo) for combo in product(*choices)}
    known = spell.known(variants)
    if not known:
        return word
    return max(known, key=lambda w: spell[w])

In [15]:
letters = ['A','B','C','D','E','F','G','H','I','J','K','L','M',
           'N','O','P','Q','R','S','T','U','V','W','X','Y','Z']

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
cap = cv2.VideoCapture(0)
cv2.namedWindow('MediaPipe Hands', cv2.WINDOW_NORMAL)
cv2.resizeWindow('MediaPipe Hands', 800, 450)
x, y, width, height = cv2.getWindowImageRect("MediaPipe Hands")
tracking = TemporalLetterTracking(letters)
motion = HandMotionDetector()
predictions = []
all_predictions = []
has_committed = False
with mp_hands.Hands(
    max_num_hands=1,
    model_complexity=0,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.4) as hands, \
mp_pose.Pose(
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.4) as pose:
        while cap.isOpened():
            success, image = cap.read()
            if not success:
              print("Ignoring empty camera frame.")
              continue
            image = cv2.flip(image, 1)
            image.flags.writeable = False
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            hand_results = hands.process(image)
            pose_results = pose.process(image)
        
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            if hand_results.multi_hand_landmarks:
                for hand_landmarks in hand_results.multi_hand_landmarks:
                    h, w = image.shape[:2]
                    x1, y1, x2, y2 = hand_bbox(hand_landmarks, w, h)
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)
                    if x2 - x1 < 10 or y2 - y1 < 10:
                        continue
                        
                    is_moving = motion.update(hand_landmarks)
                    if is_moving:
                        tracking.reset()
                        has_committed = False
                    else:
                        canvas = np.zeros((h, w, 3), dtype=np.uint8)
                        mp_drawing.draw_landmarks(
                            canvas, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style())
                        crop_landmark = canvas[y1:y2, x1:x2]
                        crop_landmark = cv2.resize(crop_landmark, (128, 128), interpolation=cv2.INTER_AREA)
                        proc = crop_landmark.reshape(1, 128, 128, 3)
                        prob = model.predict(proc, verbose=0)
                        predictions = tracking.update(prob)
                    
                        letter = LetterMade(predictions, is_moving)
                        if letter is not None and not has_committed:
                            all_predictions.append(letter)
                            has_committed = True
        
                        if predictions is not None:
                            text = ", ".join(f"{l} {t:.2f}" for l, t in predictions)
                            cv2.putText(image, text, (10, 40),
                                        cv2.FONT_HERSHEY_COMPLEX, 1, (250, 208, 92), 2)
                        else:
                            cv2.putText(image, 'NONE', (10, 40),
                                        cv2.FONT_HERSHEY_COMPLEX, 1, (250, 208, 92), 2)
                        cv2.imshow('hand_crop', crop_landmark)
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    mp_drawing.draw_landmarks(
                            image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style())
                    if pose_results.pose_landmarks:
                        ARM = [(11, 13), (13, 15)]
                        lm = pose_results.pose_landmarks.landmark
                        h, w = image.shape[:2]
                        for idx in (11, 13):
                            px, py = int(lm[idx].x * w), int(lm[idx].y * h)
                            cv2.circle(image, (px, py), 5, (0, 255, 0), -1)
                        for a, b in ARM:
                            ax, ay = int(lm[a].x * w), int(lm[a].y * h)
                            bx, by = int(lm[b].x * w), int(lm[b].y * h)
                            cv2.line(image, (ax, ay), (bx, by), (255, 255, 255), 2)
            else:
                tracking.reset()
                has_committed = False
            cv2.imshow('MediaPipe Hands', image)
            if cv2.waitKey(5) & 0xFF == 27:
              break
clumped_text = ''.join(pred[0][0] for pred in all_predictions)
bare_sentence = ReconstructSentence(clumped_text)
# final_sentence = ""
# words = bare_sentence.split(" ")
# for word in words:
#     new_word = WordVariants(word)
#     final_sentence += new_word + " "
# print(final_sentence)
print(bare_sentence)
print(all_predictions)
cap.release()
cv2.destroyAllWindows()

What
[[('W', 8.005653145883116)], [('U', 9.546485424041748)], [('H', 8.926606978056952)], [('A', 7.781266776888515)], [('T', 7.8758465051651)]]
